In [38]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_ollama import ChatOllama
import os

In [39]:
gemini_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.5-flash',
    api_key = os.environ['Gemini_api'],
    temperature = 0
)

qwen_model = ChatOllama(
    model = 'qwen2.5:1.5b',
    temperature = 0
)

In [40]:
prompt = ChatPromptTemplate.from_messages([
    ("system","You're a Helpful Assistant."),
    MessagesPlaceholder(variable_name="history"), # MessagesPlaceholder --> To insert the message indside prompt template
    ("human","{input}")
])


In [ ]:
chain = prompt | gemini_model | StrOutputParser() #LCEL

Memory = {}

def get_session_history(session_id):
    if session_id not in Memory: # if the session id not present in the Memory it allocates the new memory to the session id
        Memory[session_id] = InMemoryChatMessageHistory() #Python Memory
    return Memory[session_id] 

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history, # Here this works first --> Step 1
    input_messages_key='input',
    history_messages_key='history' # history --> stores the previous conversations
)

user1 = chain_with_memory.invoke(
    {"input" : "My name is Alice."},
    config = {"configurable":{"session_id":"user_1"}}
)

user2 = chain_with_memory.invoke(
    {"input" : "What is My Name? "},
    config = {"configurable":{"session_id":"user_1"}}
)

d:\Programs\Python\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [42]:
Memory['user_1'].messages[0].content

'My name is Alice.'

In [43]:
user1

"Hello Alice! It's nice to meet you. How can I help you today?"

In [44]:
user2

'Your name is Alice!'